In [ ]:
from libraries import *
from parameters import *
from util import *

In [ ]:
# adata = sc.read_h5ad("/home/beraslan/Projects/Abbas_screen/Data/NEPC_merged_sec_screen_RNA_scanpy_final_singlets.HVG_downstream.h5ad")
# adata_from_raw = ad.AnnData(
#     X=adata.raw.X,             # shape: (n_obs, 29546)
#     obs=adata.obs.copy(),
#     var=adata.raw.var.copy()
# )

# # (optional) carry extras that depend only on cells
# adata_from_raw.obsm = adata.obsm.copy()
# adata_from_raw.obsp = adata.obsp.copy()
# adata_from_raw.uns  = adata.uns.copy()

# adata = adata_from_raw
# adata.write("ComboScreen.h5ad")

In [ ]:
adata = sc.read_h5ad("./../Data/ComboScreen.h5ad")


In [ ]:
adata.X

In [ ]:
adata = adata[adata.obs[['ASCL1', 'KLF14', 'NEUROD1',
       'NEUROG1', 'NR3C1', 'NTC', 'SIM1', 'TET2', 'TWIST1', 'VSX1', 'ZNF385A',
       'ZNF547', 'ZNF660', 'ZNF776']].sum(axis=1) < 3]

In [ ]:
cols = ['ASCL1', 'KLF14', 'NEUROD1', 'NEUROG1', 'NR3C1', 'NTC', 'SIM1',
        'TET2', 'TWIST1', 'VSX1', 'ZNF385A', 'ZNF547', 'ZNF660', 'ZNF776']

def combine_onehot(row):
    # Select all column names where value == 1
    active = [col for col in cols if row[col] == 1]
    # Join multiple actives with '+', or return 'None' if none are active
    return '+'.join(active) if active else 'None'

adata.obs['perturbation'] = adata.obs[cols].apply(combine_onehot, axis=1)


In [ ]:
adata.obs["perturbation"].value_counts()

In [ ]:
gene_sums = np.array(adata.X.sum(axis=1)).ravel() if sp.issparse(adata.X) else adata.X.sum(axis=0)
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.hist(gene_sums, bins=100, color="skyblue", edgecolor="black")
plt.xlabel("Total number of UMIs per cell")
plt.ylabel("Number of cells")
plt.title("Distribution of number of UMIs per cell")
plt.yscale("log")  # optional, to handle long tails
plt.show()


In [ ]:
sc.pp.normalize_total(adata, target_sum=4000)
sc.pp.log1p(adata)


In [ ]:
adata

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=3000)

In [ ]:
sc.pp.scale(adata, max_value=15)

In [ ]:
sc.pp.pca(adata, n_comps=50, svd_solver='arpack')

In [ ]:
sc.pp.neighbors(adata, 
                n_neighbors=8,
                metric=par_downstream_neighbor_metric,
                n_pcs=50)

In [ ]:
sc.tl.umap(adata)

In [ ]:
adata.obs.columns

In [ ]:
for elem in ['phase','time_point','Doxo-1-differentiated_score', 'Neuroendocrine_score',
       'Intermediate-1_score', 'Differentiated-1_score',
       'Differentiated-2_score', 'Intermediate-3_score',
       'Intermediate-2_score', 'final_label',]:
    f, ax = plt.subplots(figsize=(4, 4))
    sc.pl.umap(adata, color=elem, 
           #legend_loc='on data', 
           legend_fontoutline=3, 
           legend_fontsize=14, 
           legend_fontweight='normal', 
           ax=ax, 
           show=False, 
           size=0.3)

In [ ]:
adata.obs["perturbation_time"] = (
    adata.obs["perturbation"].astype(str) + "_" + adata.obs["time_point"].astype(str)
)

In [ ]:
adata.obs["perturbation_time"]

In [ ]:
adata.obs["final_label"]

In [ ]:
import pandas as pd
import scipy.stats as stats

def test_category_enrichment(
    adata,
    col1: str,
    col2: str,
    min_count: int = 5,
    method: str = "auto",
):
    """
    Test enrichment between categories in two categorical columns of adata.obs.

    Parameters
    ----------
    adata : AnnData
        AnnData object containing the obs table.
    col1, col2 : str
        Column names in adata.obs for which enrichment is tested.
    min_count : int
        Minimum expected count per cell for chi2; otherwise Fisher’s test is used.
    method : {"auto", "fisher", "chi2"}
        Statistical test to use. "auto" picks Fisher for small tables, chi2 otherwise.

    Returns
    -------
    pd.DataFrame
        DataFrame with pairs (col1_category, col2_category) and
        odds ratio / test statistic, p-value, and counts.
    """
    df = adata.obs[[col1, col2]].dropna().copy()
    contingency = pd.crosstab(df[col1], df[col2])

    results = []
    for cat1 in contingency.index:
        for cat2 in contingency.columns:
            table = pd.DataFrame({
                "in_pair": [contingency.loc[cat1, cat2]],
                "in_col1_not_col2": [contingency.loc[cat1].sum() - contingency.loc[cat1, cat2]],
                "in_col2_not_col1": [contingency[cat2].sum() - contingency.loc[cat1, cat2]],
                "neither": [contingency.values.sum()
                            - contingency.loc[cat1].sum()
                            - contingency[cat2].sum()
                            + contingency.loc[cat1, cat2]],
            })

            # 2×2 table
            table_2x2 = [
                [table["in_pair"][0], table["in_col2_not_col1"][0]],
                [table["in_col1_not_col2"][0], table["neither"][0]],
            ]

            if method == "auto":
                expected = stats.contingency.expected_freq(table_2x2)
                chosen = "fisher" if (expected < min_count).any() else "chi2"
            else:
                chosen = method

            if chosen == "fisher":
                odds, pval = stats.fisher_exact(table_2x2, alternative="greater")
                stat = odds
            else:
                chi2, pval, _, _ = stats.chi2_contingency(table_2x2)
                stat = chi2

            results.append({
                col1: cat1,
                col2: cat2,
                "stat": stat,
                "pval": pval,
                "method": chosen,
                "count": contingency.loc[cat1, cat2],
            })

    res_df = pd.DataFrame(results)
    res_df["padj"] = stats.false_discovery_control(res_df["pval"]) if hasattr(stats, "false_discovery_control") else res_df["pval"]
    return res_df.sort_values("pval")


In [ ]:
res1=test_category_enrichment(
    adata,
    col1="perturbation_time",
    col2="final_label",
    method="fisher",
)

res1 = res1.loc[res1.padj < 0.05,]

In [ ]:
res1

In [ ]:
res2=test_category_enrichment(
    adata,
    col1="perturbation",
    col2="final_label",
    method="fisher",
)

res2 = res2.loc[res2.padj < 0.05,]

In [ ]:
res2